# Gold

## 1. Configuração

In [1]:
from pyspark.sql import functions as F

SILVER_PATH = "s3a://silver/tse"
GOLD_PATH = "s3a://gold/tse"

print(f"Silver: {SILVER_PATH}")
print(f"Gold: {GOLD_PATH}")

Silver: s3a://silver/tse
Gold: s3a://gold/tse


## 2. Carregamento das tabelas Silver

In [2]:
candidatura = spark.read.format("delta").load(f"{SILVER_PATH}/candidatura")
partido = spark.read.format("delta").load(f"{SILVER_PATH}/partido")
cargo = spark.read.format("delta").load(f"{SILVER_PATH}/cargo")
municipio = spark.read.format("delta").load(f"{SILVER_PATH}/municipio")
instrucao = spark.read.format("delta").load(f"{SILVER_PATH}/instrucao")
bens = spark.read.format("delta").load(f"{SILVER_PATH}/bens")

print("Tabelas carregadas com sucesso.")

26/06/21 22:56:58 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Tabelas carregadas com sucesso.


## 3. fato_candidatura_dashboard

In [3]:
fato_candidatura_dashboard = (
    candidatura
    .select(
        F.col("id_candidatura").alias("sq_candidato"),
        F.col("id_cargo_fk").alias("codigo_cargo"),
        F.col("id_partido_fk").alias("numero_partido"),
        F.col("id_municipio_fk").alias("sigla_ue"),
        F.col("id_grau_instrucao_fk").alias("codigo_grau_instrucao"),
        F.col("id_ocupacao_fk").alias("codigo_ocupacao"),
        F.col("genero").alias("codigo_genero"),
        F.col("cor_raca").alias("descricao_cor_raca")
    )
)

fato_candidatura_dashboard.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}/fato_candidatura_dashboard")

print("fato_candidatura_dashboard criada")

26/06/21 22:57:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/06/21 22:57:07 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

fato_candidatura_dashboard criada


## 4. fato_bem_candidato_dashboard

In [4]:
fato_bem_candidato_dashboard = (
    bens
    .select(
        F.col("id_candidatura_fk").alias("sq_candidato"),
        F.col("id_tipo_bem_fk").alias("codigo_tipo_bem_candidato"),
        F.col("DESCRICAO_detalhada").alias("descricao_bem_candidato"),
        F.col("valor").cast("double").alias("valor_bem_candidato")
    )
)

fato_bem_candidato_dashboard.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}/fato_bem_candidato_dashboard")

print("fato_bem_candidato_dashboard criada")

fato_bem_candidato_dashboard criada


## 5. dim_cargo

In [5]:
dim_cargo = (
    cargo
    .select(
        F.col("id_cargo").alias("codigo_cargo"),
        F.col("descricao").alias("descricao_cargo")
    )
)

dim_cargo.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}/dim_cargo")

print("dim_cargo criada")

dim_cargo criada


## 6. dim_partido

In [6]:
dim_partido = (
    partido
    .select(
        F.col("id_partido").alias("numero_partido"),
        F.col("sigla").alias("sigla_partido"),
        F.col("nome").alias("nome_partido")
    )
)

dim_partido.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}/dim_partido")

print("dim_partido criada")

dim_partido criada


## 7. dim_municipio

In [7]:
dim_municipio = (
    municipio
    .select(
        F.col("sigla_uf"),
        F.col("id_municipio").alias("sigla_ue"),
        F.col("nome").alias("nome_ue")
    )
)

dim_municipio.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{GOLD_PATH}/dim_municipio")

print("dim_municipio criada")

dim_municipio criada


## 8. Validação

In [8]:
print("=== CAMADA GOLD ===")

for tabela in [
    "fato_candidatura_dashboard",
    "fato_bem_candidato_dashboard",
    "dim_cargo",
    "dim_partido",
    "dim_municipio"
]:
    total = (
        spark.read.format("delta")
        .load(f"{GOLD_PATH}/{tabela}")
        .count()
    )

    print(f"{tabela}: {total:,} registros")

=== CAMADA GOLD ===
fato_candidatura_dashboard: 78,349 registros
fato_bem_candidato_dashboard: 156,470 registros
dim_cargo: 3 registros
dim_partido: 30 registros
dim_municipio: 645 registros


## 9. Amostra da fato principal

In [9]:
spark.read.format("delta") \
    .load(f"{GOLD_PATH}/fato_candidatura_dashboard") \
    .show(5, truncate=False)

+------------+------------+--------------+--------+---------------------+---------------+-------------+------------------+
|sq_candidato|codigo_cargo|numero_partido|sigla_ue|codigo_grau_instrucao|codigo_ocupacao|codigo_genero|descricao_cor_raca|
+------------+------------+--------------+--------+---------------------+---------------+-------------+------------------+
|250002263326|13          |50            |71790   |3                    |999            |2            |PARDA             |
|250002121814|11          |15            |63630   |6                    |275            |2            |BRANCA            |
|250002140532|13          |20            |63711   |6                    |257            |4            |PARDA             |
|250001904055|13          |70            |70378   |6                    |999            |4            |BRANCA            |
|250002343269|13          |70            |64777   |8                    |298            |2            |BRANCA            |
+------------+--